<a href="https://colab.research.google.com/github/mahigarg0403-blip/Customer-purchase-prediction_mahi_bhumika/blob/main/notebooks/%20step4_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
#cell 1 : mount drive
from google.colab import drive
drive.mount('/content/drive',force_remount = True)

Mounted at /content/drive


In [84]:
#cell 2 : importing libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
print("imported!")

imported!


In [85]:
#cell 3 : importing the datasets
x_train_lr = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/x_train_lr.csv')
x_test_lr = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/x_test_lr.csv')
y_train_lr = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/y_train_lr.csv')
y_test_lr = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/y_test_lr.csv')

x_train_tree = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/x_train_tree.csv')
x_test_tree = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/x_test_tree.csv')
y_train_tree = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/y_train_tree.csv')
y_test_tree = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step2/y_test_tree.csv')

cleaned_df = pd.read_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/cleaned_data.csv')

print("Datasets Imported!")

Datasets Imported!


In [86]:
#cell 4 : Dropping BounceRates column
print(cleaned_df[['BounceRates','Revenue']].corr())
print(cleaned_df[['ExitRates','Revenue']].corr())
#higher correlation with ExitRates thus we drop BounceRates
for x in (x_train_lr,x_train_tree,x_test_lr,x_test_tree):
  x.drop(columns = ['BounceRates'], inplace = True)

print("BounceRates column dropped!")

             BounceRates   Revenue
BounceRates     1.000000 -0.145091
Revenue        -0.145091  1.000000
           ExitRates  Revenue
ExitRates    1.00000 -0.20432
Revenue     -0.20432  1.00000
BounceRates column dropped!


In [87]:
#cell 5 : binning PageValues Column

def bin_pagevalues_lr(val) :
  if val == 0:
    return 0
  elif val <= x_train_lr["PageValues"][x_train_lr['PageValues'] > 0].median() :
    return 1
  else:
    return 2

x_train_lr['PageValues_bin'] = x_train_lr['PageValues'].apply(bin_pagevalues_lr)
x_test_lr['PageValues_bin'] = x_test_lr['PageValues'].apply(bin_pagevalues_lr)

def bin_pagevalues_tree(val) :
  if val == 0:
    return 0
  elif val <= x_train_tree["PageValues"][x_train_tree['PageValues'] > 0].median() :
    return 1
  else:
    return 2

x_train_tree['PageValues_bin'] = x_train_tree['PageValues'].apply(bin_pagevalues_tree)
x_test_tree['PageValues_bin'] = x_test_tree['PageValues'].apply(bin_pagevalues_tree)

print("PageValues_bin column added!")

PageValues_bin column added!


In [88]:
#cell 6 : Log transformation for ProductRelated_duration
x_train_lr['ProductRelated_Duration_log'] = np.log1p(x_train_lr['ProductRelated_Duration'])
x_test_lr['ProductRelated_Duration_log']  = np.log1p(x_test_lr['ProductRelated_Duration'])
x_train_tree['ProductRelated_Duration_log'] = np.log1p(x_train_tree['ProductRelated_Duration'])
x_test_tree['ProductRelated_Duration_log']  = np.log1p(x_test_tree['ProductRelated_Duration'])
print("ProductRelated_Duration_log column added!")

ProductRelated_Duration_log column added!


In [89]:
#cell 7 : Interaction_Value = (ProductRelated x PageValues)
for x in (x_train_lr,x_train_tree,x_test_lr,x_test_tree):
  x['Interaction_Value'] = x['ProductRelated'] * x['PageValues']
print("Interaction_Value column added!")

Interaction_Value column added!


In [91]:
# cell 8 : Session_Intensity = (ProductRelated_Duration / ProductRelated)
for x in (x_train_lr, x_train_tree, x_test_lr, x_test_tree):
    # If ProductRelated is not 0, do the division; otherwise, set it to 0
    x['Session_Intensity'] = np.where(
        x['ProductRelated'] != 0,
        x['ProductRelated_Duration'] / x['ProductRelated'],
        0)

print("Session_Intensity column added!")

Session_Intensity column added!


In [92]:
#cell 9 : checking highest grossing month
cleaned_df.groupby('Month')['Revenue'].mean()

,Revenue
Month,
2,0.016575
3,0.103226
5,0.109643
6,0.101754
7,0.152778
8,0.175520
9,0.191964
10,0.209472
11,0.254863


In [93]:
#cell 10 : for month 11 (november) - checking highest visitorType
cleaned_df[cleaned_df['Month']==11].groupby('VisitorType')['Revenue'].mean()

,Revenue
VisitorType,
0,0.305489
1,0.136364
2,0.247540


In [94]:
#cell 11 : November_New binary column
x_train_lr['November_New'] = ((x_train_lr['Month_Nov']==1) & (x_train_lr['VisitorType_Other']==0) & (x_train_lr['VisitorType_Returning_Visitor']==0)).astype(int)
x_test_lr['November_New'] = ((x_test_lr['Month_Nov']==1) & (x_test_lr['VisitorType_Other'] == 0) & (x_test_lr['VisitorType_Returning_Visitor'] == 0)).astype(int)
x_train_tree['November_New'] = ((x_train_tree['Month'] == 11) & (x_train_tree['VisitorType'] == 0)).astype(int)
x_test_tree['November_New'] = ((x_test_tree['Month'] == 11) & (x_test_tree['VisitorType'] == 0)).astype(int)
print("November_New column added!")

November_New column added!


In [95]:
#cell 12 : scaling LR data
num_cols = ['Administrative_Duration','Informational_Duration','ProductRelated_Duration',
            'ExitRates','PageValues','SpecialDay']
scaler = StandardScaler()
x_train_lr[num_cols] = scaler.fit_transform(x_train_lr[num_cols])
x_test_lr[num_cols]  = scaler.transform(x_test_lr[num_cols])
print("Scaling Done!")

Scaling Done!


In [96]:
#cell 13 : SMOTE on Training data for fixing class imbalance
smote = SMOTE(random_state = 42)
x_train_lr_final, y_train_lr_final = smote.fit_resample(x_train_lr, y_train_lr)
x_train_tree_final, y_train_tree_final= smote.fit_resample(x_train_tree, y_train_tree)
print("SMOTE done!")

SMOTE done!


In [97]:
#cell 14 : Saving final datasets
x_train_lr_final.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/x_train_lr_final.csv', index = False)
x_test_lr.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/x_test_lr_final.csv', index = False)
y_train_lr_final.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/y_train_lr_final.csv', index = False)
y_test_lr.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/y_test_lr_final.csv', index = False)
x_train_tree_final.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/x_train_tree_final.csv', index = False)
x_test_tree.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/x_test_tree_final.csv', index = False)
y_train_tree_final.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/y_train_tree_final.csv', index = False)
y_test_tree.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/step 4/y_test_tree_final.csv', index = False)
print("all saved")

all saved
